In [1]:
import os

In [2]:
%pwd

'c:\\projects\\SellWise\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\projects\\SellWise'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    calendar_path: Path
    sell_prices_path: Path
    sales_path: Path

In [6]:
from SellWise.constants import *
from SellWise.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath=CONFIG_FILE_PATH, 
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([(self.config.artifacts_root)])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
            calendar_path=config.calendar_path,
            sell_prices_path=config.sell_prices_path,
            sales_path=config.sales_path
        )
        
        return data_ingestion_config

In [8]:
import os
import urllib.request as request
import zipfile
from SellWise import logger
from SellWise.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Unzipped file in dir: {unzip_path}")

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-09-22 13:43:50,257: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-22 13:43:50,263: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-22 13:43:50,265: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-22 13:43:50,267: INFO: common: created directory at: artifacts]
[2026-09-22 13:43:50,269: INFO: common: created directory at: artifacts/data_ingestion]
[2026-09-22 13:43:53,651: INFO: 1990249451: artifacts/data_ingestion/data.zip downloaded! with following info: 
Content-Type: application/binary
Accept-Ranges: bytes
Cache-Control: max-age=60
Content-Disposition: attachment; filename="dataset.zip"; filename*=UTF-8''dataset.zip
Content-Security-Policy: sandbox
Etag: 1790063486878545d
Pragma: public
Referrer-Policy: no-referrer
Vary: Origin
X-Content-Security-Policy: sandbox
X-Content-Type-Options: nosniff
X-Robots-Tag: noindex, nofollow, noimageindex
X-Server-Response-Time: 244
X-Webkit-Csp: sandbox
Date: Tue, 22 Sep 2026 08